In [1]:
from metadrive.engine.asset_loader import AssetLoader
import os
from metadrive.scenario import utils as sd_utils
import numpy as np
from pyquaternion import Quaternion
import torch
import lance
import os
import random
import pickle
import copy
import PIL
import time
import copy
import lance
import lance.torch.data
from typing import TypedDict
import numpy as np
from scipy.spatial.transform import Rotation as R
from lance.sampler import ShardedFragmentSampler
import PIL.Image as PImage
import torchvision
from metadrive.scenario import ScenarioDescription
import sys
sys.path.append("..")
from nuscenes_tools.nuscenes_utils.pipelines import Compose
from nuscenes_tools.nuscenes_utils.box3d_instance import LiDARInstance3DBoxes
import nuscenes_tools.nuscenes_utils.pipelines as pipelines
from infinity.utils.vis_utils import draw_box_on_imgs

import scenarionet_tools.storage as storage

import logging


In [2]:
import json
creds = json.load(open('credentials.json'))['nuscenes']
os.environ['AWS_ACCESS_KEY_ID'] = creds['AWS_ACCESS_KEY_ID']
os.environ['AWS_ENDPOINT_URL'] = creds['AWS_ENDPOINT_URL']
os.environ['AWS_SECRET_ACCESS_KEY'] = creds['AWS_SECRET_ACCESS_KEY']
os.environ['AWS_DEFAULT_REGION'] = creds['AWS_DEFAULT_REGION']
os.environ['PREAUTH_URL'] = creds['PREAUTH_URL']


In [3]:
# nuscenes_scenarionet_nuplan_coord, nuscenes_scenarionet
dataset_path = "/media/training_data/nuscenes_golden/nuscenes_scenarionet_debug_1.5_pos/"
nuscenes_data = AssetLoader.file_path(dataset_path, unix_style=False) # Use the built-in datasets with simulator
# nuscenes_data = AssetLoader.file_path("/media/training_data/nuscenes_golden", "nuscenes_scenarionet", unix_style=False) # Use the built-in datasets with simulator

os.listdir(nuscenes_data) # there are 10 nuscenes scenario file with a 'dataset_summary.pkl' and a 'dataset_summary.pkl'

['nuscenes_scenarionet_debug_1.5_pos_4',
 'nuscenes_scenarionet_debug_1.5_pos_0',
 'dataset_summary.pkl',
 'nuscenes_scenarionet_debug_1.5_pos_1',
 'nuscenes_scenarionet_debug_1.5_pos_3',
 'dataset_mapping.pkl',
 'nuscenes_scenarionet_debug_1.5_pos_5',
 'nuscenes_scenarionet_debug_1.5_pos_6',
 'nuscenes_scenarionet_debug_1.5_pos_7',
 'nuscenes_scenarionet_debug_1.5_pos_2']

In [4]:
dataset_summary = sd_utils.read_dataset_summary(dataset_path)
print(f"The dataset summary is a {type(dataset_summary)}, with lengths {len(dataset_summary)}.")


The dataset summary is a <class 'tuple'>, with lengths 3.


In [5]:
scenario_summary, scenario_ids, scenario_files = dataset_summary


print(f"The scenario summary is a dict with keys: {scenario_summary.keys()} \nwhere each value of the dict is the summary of a scenario.\n")

The scenario summary is a dict with keys: dict_keys(['sd_nuscenes_v1.0-mini_scene-0061.pkl', 'sd_nuscenes_v1.0-mini_scene-0103.pkl', 'sd_nuscenes_v1.0-mini_scene-0553.pkl', 'sd_nuscenes_v1.0-mini_scene-0655.pkl', 'sd_nuscenes_v1.0-mini_scene-0757.pkl', 'sd_nuscenes_v1.0-mini_scene-0796.pkl', 'sd_nuscenes_v1.0-mini_scene-0916.pkl', 'sd_nuscenes_v1.0-mini_scene-1077.pkl', 'sd_nuscenes_v1.0-mini_scene-1094.pkl', 'sd_nuscenes_v1.0-mini_scene-1100.pkl']) 
where each value of the dict is the summary of a scenario.



In [6]:
print(scenario_ids)
print(len(scenario_ids))

['sd_nuscenes_v1.0-mini_scene-0061.pkl', 'sd_nuscenes_v1.0-mini_scene-0103.pkl', 'sd_nuscenes_v1.0-mini_scene-0553.pkl', 'sd_nuscenes_v1.0-mini_scene-0655.pkl', 'sd_nuscenes_v1.0-mini_scene-0757.pkl', 'sd_nuscenes_v1.0-mini_scene-0796.pkl', 'sd_nuscenes_v1.0-mini_scene-0916.pkl', 'sd_nuscenes_v1.0-mini_scene-1077.pkl', 'sd_nuscenes_v1.0-mini_scene-1094.pkl', 'sd_nuscenes_v1.0-mini_scene-1100.pkl']
10


In [7]:
example_scenario_summary = scenario_summary[scenario_ids[0]]
print(f"The summary of a scenario is a dict with keys: {example_scenario_summary.keys()}")

The summary of a scenario is a dict with keys: dict_keys(['dataset', 'metadrive_processed', 'map', 'date', 'timeofday', 'coordinate', 'id', 'scenario_id', 'sample_rate', 'ts', 'sdc_id', 'raw_sensors_valid', 'object_summary', 'number_summary'])


In [8]:
# A string indicating the coordinate system of the scenario.
print(example_scenario_summary["coordinate"])

right-handed


In [9]:
# A list containing the wall time of each time step
print(example_scenario_summary["ts"])

[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5, 15.0, 15.5, 16.0, 16.5, 17.0, 17.5, 18.0, 18.5, 19.0]


In [10]:
# A boolean indicator saying whether the scenario is exported from MetaDrive.
# Note: we have a replay and record systems in MetaDrive that will generate 
# scenarios with `metadrive_processed=True`.
print(example_scenario_summary["metadrive_processed"])

False


In [11]:
# Where the scenario comes from
print(example_scenario_summary["dataset"])

nuscenes


In [12]:
# The ID of the scenario
print(example_scenario_summary["scenario_id"])

scene-0061


In [13]:
scenario_pkl_file = scenario_ids[0]
print(scenario_pkl_file)

sd_nuscenes_v1.0-mini_scene-0061.pkl


In [14]:
# Get the relative path to the .pkl file
print("The pkl file relative path: ", scenario_files[scenario_pkl_file]) 

# Get the absolute path to the .pkl file
abs_path_to_pkl_file = os.path.join(dataset_path, scenario_files[scenario_pkl_file], scenario_pkl_file)
print("The pkl file absolute path: ", abs_path_to_pkl_file)

The pkl file relative path:  nuscenes_scenarionet_debug_1.5_pos_0
The pkl file absolute path:  /media/training_data/nuscenes_golden/nuscenes_scenarionet_debug_1.5_pos/nuscenes_scenarionet_debug_1.5_pos_0/sd_nuscenes_v1.0-mini_scene-0061.pkl


In [15]:
# Call utility function in MD and get the Scenario Description object
scenario = sd_utils.read_scenario_data(abs_path_to_pkl_file)

print(f"\nThe raw data type after reading the .pkl file is {type(scenario)}")


The raw data type after reading the .pkl file is <class 'metadrive.scenario.scenario_description.ScenarioDescription'>


In [16]:
print(f"The keys in a ScenarioDescription are: {scenario.keys()}")


The keys in a ScenarioDescription are: dict_keys(['id', 'version', 'length', 'metadata', 'tracks', 'dynamic_map_states', 'map_features', 'raw_sensors'])


In [17]:
scenario['id']

'scene-0061'

In [18]:
print(print(scenario['metadata'].keys()))

dict_keys(['dataset', 'metadrive_processed', 'map', 'date', 'timeofday', 'coordinate', 'id', 'scenario_id', 'sample_rate', 'ts', 'sdc_id', 'raw_sensors_valid', 'object_summary', 'number_summary'])
None


In [19]:
print(scenario['metadata']['object_summary'])

{'d7f9aa76f8164fd7b043da901503ef16': {'type': 'pedestrian', 'object_id': 'd7f9aa76f8164fd7b043da901503ef16', 'track_length': 39, 'moving_distance': 9.64953204592776, 'valid_length': 14, 'continuous_valid_length': 14}, '01c083253f30417dbe374f84e8ba6c19': {'type': 'barrier', 'object_id': '01c083253f30417dbe374f84e8ba6c19', 'track_length': 39, 'moving_distance': 0.8451870356680776, 'valid_length': 39, 'continuous_valid_length': 39}, '31cf67656bb7486293cf992bc285c225': {'type': 'traffic_cone', 'object_id': '31cf67656bb7486293cf992bc285c225', 'track_length': 39, 'moving_distance': 0.0, 'valid_length': 1, 'continuous_valid_length': 1}, 'e045fa4da9e64d759fb780b20c707b3e': {'type': 'traffic_cone', 'object_id': 'e045fa4da9e64d759fb780b20c707b3e', 'track_length': 39, 'moving_distance': 0.004472135955097197, 'valid_length': 2, 'continuous_valid_length': 2}, '62bfd58b7fc145e6b3c4f30552f7253f': {'type': 'traffic_cone', 'object_id': '62bfd58b7fc145e6b3c4f30552f7253f', 'track_length': 39, 'moving_dis

In [20]:

def convert_bbox(xyz, heading, rotation, translation):
    is_torch = torch.is_tensor(xyz)
    if is_torch:
        device = xyz.device
        xyz = xyz.numpy()
        heading = heading.numpy()
        rotation = rotation.numpy()
        translation = translation.numpy()
    xyz =  xyz @ rotation.T + translation[None]
    # heading = np.stack([np.zeros_like(heading), np.zeros_like(heading), heading], axis=1)
    rot = R.from_matrix(rotation[None]).as_euler('xyz', degrees=False)
    
    heading = heading + rot[..., -1]
    
    # heading = R.from_euler('xyz', heading, degrees=False).as_matrix()
    # heading = rotation[None] @ heading
    # heading = R.from_matrix(heading).as_euler('xyz', degrees=False)
    # heading = heading[..., -1]
    
    if is_torch:
        xyz = torch.from_numpy(xyz).to(device)
        heading = torch.from_numpy(heading).to(device)
    return xyz, heading


def to_tensor(batch, **kwargs):
    return batch



def _fetch_next(data_iter, data_path, rank, world_size):
    if data_iter == None:
        lance_dataset = lance.torch.data.LanceDataset(
            data_path,
            columns=["scenario_id", "scenario"],
            filter="raw_sensors_valid = true",
            batch_size=1,
            # batch_readahead=8,  # Control multi-threading reads.
            to_tensor_fn=to_tensor,
            sampler = ShardedFragmentSampler(
                rank=rank,  # Rank of the current dataloader thread
                world_size=world_size,  # Total number of processes
                randomize=True,
                seed=random.randint(0, 100000),
            )
        )
        data_iter = iter(lance_dataset)
    try:
        output = next(data_iter)
    except StopIteration:
        lance_dataset = lance.torch.data.LanceDataset(
            data_path,
            columns=["scenario_id", "scenario"],
            filter="raw_sensors_valid = true",
            batch_size=1,
            # batch_readahead=8,  # Control multi-threading reads.
            to_tensor_fn=to_tensor,
            sampler = ShardedFragmentSampler(
                rank=rank,  # Rank of the current dataloader thread
                world_size=world_size,  # Total number of processes
                randomize=True,
                seed=random.randint(0, 100000),
            )
        )
        data_iter = iter(lance_dataset)
        output = next(data_iter)
    return output, data_iter



def get_nextitem(data_iter, data_path, rank, world_size):
    max_retries = 10
    for attempt in range(max_retries):
        try:
            output, data_iter = _fetch_next(data_iter, data_path, rank, world_size)
            break  # Success, exit the loop
        except (OSError, ValueError) as e:
            if "429 Too Many Requests" in str(e):
                if attempt < max_retries - 1:
                    sleep_time = (2**attempt) + random.uniform(0, 1)
                    print(
                        f"WARNING: Received 429 Too Many Requests, retrying in {sleep_time:.2f} seconds...", flush=True
                    )
                    time.sleep(sleep_time)
                else:
                    print(
                        "ERROR: Max retries reached for loading dataset from S3.", flush=True
                    )
                    raise  # Re-raise the error after max retries
            else:
                raise  # Re-raise if not a 429 error
    return output, data_iter

class NuScenesataWrapper(torch.utils.data.IterableDataset):
    def __init__(self, data_config, rank=0, world_size=1, pipeline=None, leaveout=True):
        self.data_config = data_config
        self.data_path = data_config['dataset_path']
        self.leaveout = leaveout

        if pipeline is not None:
            self.pipeline = Compose(pipeline)
        else:
            self.pipeline = None
        
        self.rank = rank
        self.world_size = world_size
        
        if 'load_action' in self.data_config and self.data_config['load_action']:
            with open(self.data_config['kmeans_path'], 'rb') as file:
                self.kmeans = pickle.load(file)

    def _get_scenario_info(self, scenario):
        scenario_dict = {}
        date = scenario['metadata']['date']
        scenario_dict['timeofday'] = date
        
        scenario_dict['location'] = scenario['metadata']['map']
        
        objects = scenario['metadata']['object_summary']
        sdc_id = scenario['metadata']['sdc_id']
        object_categories = []
        for obj_id in objects:
            if sdc_id != obj_id:
                obj_cat = objects[obj_id]['type'].lower()
                if obj_cat not in object_categories:
                    object_categories.append(obj_cat)
        object_categories = ', '.join(object_categories)
        description = object_categories
        scenario_dict['description'] = description

        if 'language_description' in scenario:
            scenario_dict['language_description'] = scenario['language_description']

        return scenario_dict
    
    def _get_object_info(self, scenario, id):
        gt_bboxes_3d = []
        gt_names_3d = []
        
        sdc_id = scenario['metadata']['sdc_id']
        tracks = scenario['tracks']
        for obj_id in tracks:
            if obj_id == 'sdc_id':
                continue
            obj_info = tracks[obj_id]
            if not obj_info['state']['valid'][id]:
                continue
            position = obj_info['state']['position'][id]
            heading = obj_info['state']['heading'][id][None]
            velocity = obj_info['state']['velocity'][id]
            length = obj_info['state']['length'][id]
            width = obj_info['state']['width'][id]
            height = obj_info['state']['height'][id]
            
            obj_size = np.concatenate([length, width, height], axis=0)
            
            bbox = np.concatenate([position, obj_size, heading], axis=0)
            gt_bboxes_3d.append(bbox)

            label = obj_info['type']
            gt_names_3d.append(label)
        
        if len(gt_bboxes_3d) > 0:
            gt_bboxes_3d = np.stack(gt_bboxes_3d, axis=0)
        else:
            gt_bboxes_3d = np.zeros((0, 7))
        gt_names_3d = np.array(gt_names_3d)

        return {'gt_bboxes_3d': gt_bboxes_3d, 'gt_names_3d': gt_names_3d}
        
    
    def _get_camera_info(self, camera_sensors):
        all_camera_info = {}
        
        for cam_name in camera_sensors:
            cam_data = camera_sensors[cam_name]
            cam_info = {}
            cam_info['data_path'] = cam_data['cam_abs_path'] # cam_abs_path in s3
            # cam_info['data_path'] = cam_info['data_path'].replace('s3://', '/media/')
            # cam_info['data_path'] = cam_info['data_path'].replace('s3://', '/media/')

            cam_info['cam_intrinsic'] = np.array(cam_data['cam_intrinsic'], dtype=np.float32)
            cam_info['sensor2lidar_translation'] = np.array(cam_data['sensor2lidar_translation'], dtype=np.float32)
            cam_info['sensor2lidar_rotation'] = np.array(cam_data['sensor2lidar_rotation'], dtype=np.float32)

            cam_info['ego2global_rotation'] = Quaternion(np.array(cam_data['ego2global_rotation'], dtype=np.float32)).rotation_matrix
            cam_info['ego2global_translation'] = np.array(cam_data['ego2global_translation'], dtype=np.float32)

            sensor2lidar = np.eye(4)
            sensor2lidar[:3, :3] = cam_info['sensor2lidar_rotation']
            sensor2lidar[:3, 3] = cam_info['sensor2lidar_translation']
            
            ego2global = np.eye(4)
            ego2global[:3, :3] = cam_info['ego2global_rotation']
            ego2global[:3, 3] = cam_info['ego2global_translation']
            
            sensor2ego = np.eye(4)
            sensor2ego[:3, :3] = Quaternion(np.array(cam_data['sensor2ego_rotation'], dtype=np.float32)).rotation_matrix
            sensor2ego[:3, 3] = np.array(cam_data['sensor2ego_translation'], dtype=np.float32)

            global2lidar = sensor2lidar @ np.linalg.inv(sensor2ego) @ np.linalg.inv(ego2global)
            cam_info['global2lidar_rotation'] = global2lidar[:3, :3]
            cam_info['global2lidar_translation'] = global2lidar[:3, 3]

            all_camera_info[cam_name] = cam_info
            

        return all_camera_info

    def _get_action_info(self, raw_sensors, id, weight=[1, 1, 10]):
        action_horizon = self.data_config['action_horizon']
        seq_len = len(raw_sensors)
        
        empty_action_dict = {
            'action_cluster_id': -1,
            'future_trajectory': torch.zeros([action_horizon, 4]),
            'action_cluster_center': torch.zeros([action_horizon, 3])
        }
        if seq_len - id <= action_horizon:
            return empty_action_dict
        
        ego2global_rt_list = []
        for i in range(id, id+action_horizon+1):
            if len(raw_sensors[i]['images']) == 0:
                return empty_action_dict
            for cam in self.data_config['cams']:
                if cam in raw_sensors[i]['images']:
                    current_cam = raw_sensors[i]['images'][cam]
                    break
            ego2global_rotation = Quaternion(current_cam['ego2global_rotation']).rotation_matrix
            ego2global_translation = current_cam['ego2global_translation']
            
            ego2global_rt = np.eye(4)
            ego2global_rt[:3, :3] = ego2global_rotation
            ego2global_rt[:3, 3] = ego2global_translation

            ego2global_rt_list.append(ego2global_rt)
        
        ego2global_rts = np.stack(ego2global_rt_list, axis=0)
        ego2first_rts = np.linalg.inv(ego2global_rts[0:1]) @ ego2global_rts
        
        ego2first_rts = ego2first_rts[1:]
        traj_xyz = ego2first_rts[:, :3, 3]
        traj_yaw = R.from_matrix(ego2first_rts[:, :3, :3]).as_euler('xyz', degrees=False)[:, 2:3]
        
        traj = np.concatenate([traj_xyz[..., :2], traj_yaw], axis=1)
        
        traj_weighted = traj * np.array(weight)[None]
        traj_weighted = traj_weighted.reshape(1, -1)
        
        cluster_id = self.kmeans.predict(traj_weighted)[0]
        cluster_center = copy.deepcopy(self.kmeans.cluster_centers_[cluster_id])
        cluster_center = cluster_center.reshape(-1, 3)
        cluster_center = cluster_center / np.array(weight)[None]

        traj = np.concatenate([traj_xyz, traj_yaw], axis=1)
        action_dict = {
            'action_cluster_id': int(cluster_id),
            'future_trajectory': torch.from_numpy(traj),
            'action_cluster_center': torch.from_numpy(cluster_center),
        }
        
        return action_dict

    def _get_sensor_sequence_info(self, scenario):
        horizon = self.data_config['horizon']
        if horizon == 'all':
            assert not self.data_config['random_interval']
            horizon = len(scenario['raw_sensors']) // self.data_config['interval']
        intervals = []
        if self.data_config['random_interval']:
            intervals = [random.randint(0, self.data_config['interval']) for _ in range(horizon-1)]
        else:
            intervals = [self.data_config['interval']] * (horizon - 1)
        
        intervals = [0] + intervals
        
        raw_sensors = scenario['raw_sensors']
        total_len = sum(intervals)
        sids = list(range(len(raw_sensors)-total_len))
        sid = random.choice(sids)
        if 'first_frame' in self.data_config:
            sid = self.data_config['first_frame']

        timesteps = np.cumsum(intervals)
        seq_ids = timesteps + sid
        assert len(seq_ids) == horizon and seq_ids[-1] < len(raw_sensors)

        image_dict_sequence = []
        bbox_sequence = []
        action_sequence = []
        for id in seq_ids:
            all_camera_info = self._get_camera_info(raw_sensors[id]['images'])
            image_dict_sequence.append(all_camera_info)

            if 'load_bbox' in self.data_config and self.data_config['load_bbox']:
                all_object_info = self._get_object_info(scenario, id)
                bbox_sequence.append(all_object_info)
            
            if 'load_action' in self.data_config and self.data_config['load_action'] and len(all_camera_info) == self.data_config['Ncams']:
                action_info = self._get_action_info(raw_sensors, id)
                action_sequence.append(action_info)


        timesteps = timesteps / ((horizon - 1) * self.data_config['interval'])
        return image_dict_sequence, bbox_sequence, action_sequence, timesteps, seq_ids

    def _filter_bbox(self, xyz):
        obj_range = self.data_config['object_range']
        flag = (xyz[:, 0] > obj_range[0]) & (xyz[:, 1] > obj_range[1]) & (xyz[:, 2] > obj_range[2]) \
            & (xyz[:, 0] < obj_range[3]) & (xyz[:, 1] < obj_range[4]) & (xyz[:, 2] < obj_range[5])
        return flag        

    def _convert_and_filter_bbox_coordinate(self, bbox_sequence, image_dict_sequence):
        first_frame_info = image_dict_sequence[0]
        first_frame_global2lidar_rotation = first_frame_info['CAM_FRONT']['global2lidar_rotation']
        first_frame_global2lidar_translation = first_frame_info['CAM_FRONT']['global2lidar_translation']

        for frame_box, frame_info in zip(bbox_sequence, image_dict_sequence):
            if len(frame_box['gt_bboxes_3d']) == 0:
                continue
            curr_frame_global2lidar_rotation = frame_info['CAM_FRONT']['global2lidar_rotation']
            curr_frame_global2lidar_translation = frame_info['CAM_FRONT']['global2lidar_translation']
            
            xyz = frame_box['gt_bboxes_3d'][:, :3]
            heading = frame_box['gt_bboxes_3d'][:, 6]

            curr_xyz, _ = convert_bbox(xyz, heading, curr_frame_global2lidar_rotation, curr_frame_global2lidar_translation)
            flag = self._filter_bbox(curr_xyz)
            frame_box['gt_bboxes_3d'] = frame_box['gt_bboxes_3d'][flag]
            frame_box['gt_names_3d'] = frame_box['gt_names_3d'][flag]

            if len(frame_box['gt_bboxes_3d']) == 0:
                continue
            new_xyz, new_heading = convert_bbox(frame_box['gt_bboxes_3d'][:, :3], frame_box['gt_bboxes_3d'][:, 6], first_frame_global2lidar_rotation, first_frame_global2lidar_translation)
            frame_box['gt_bboxes_3d'][:, :3] = new_xyz
            frame_box['gt_bboxes_3d'][:, 6] = new_heading
            
            frame_box['gt_bboxes_3d'] = LiDARInstance3DBoxes(frame_box['gt_bboxes_3d'], origin=(0.5, 0.5, 0.0))

        return bbox_sequence

    def _preprocess_sample(self, item):
        scenario = pickle.loads(item['scenario'])
        scenario = ScenarioDescription(scenario)
        return self._get_frame_from_scenario(scenario, item['scenario_id'])
        
    def _get_frame_from_scenario(self, scenario, scenario_id=None):

        if scenario_id is None:
            if "id" in scenario:
                scenario_id = scenario["id"]
        data_info = {}
        # data_info.update(scenario)
        scenario_dict = self._get_scenario_info(scenario)
        data_info.update(scenario_dict)
        data_info['sample_idx'] = scenario_id
        
        image_dict_sequence, bbox_sequence, action_sequence, timesteps, seq_ids = self._get_sensor_sequence_info(scenario)
        for camera_infos in image_dict_sequence:
            if len(camera_infos) < self.data_config['Ncams']:
                return None
        
        if 'load_bbox' in self.data_config and self.data_config['load_bbox']:
            bbox_sequence = self._convert_and_filter_bbox_coordinate(bbox_sequence, image_dict_sequence)
        
        frame_dict_sequence = []
        for fid, camera_infos in enumerate(image_dict_sequence):
            frame_dict = {}
            frame_data_info = data_info.copy()
            if 'language_description' in frame_data_info:
                try:
                    description = frame_data_info['language_description'][seq_ids[fid]]['description']
                    sentences = description.split('.')
                    if sentences[-1] == '':
                        sentences = sentences[:-1]
                    sentences = [item for item in sentences if random.random() > 0.25]
                    description = '.'.join(sentences) + '.'
                    frame_data_info['description'] = description
                except:
                    print(scenario_id)
                    print(frame_data_info['language_description'][seq_ids[fid]])
                    frame_data_info['description'] = ''


            frame_dict.update(frame_data_info)
            frame_dict['img_info'] = camera_infos
            frame_dict['timestep'] = timesteps[fid]
            
            if 'load_bbox' in self.data_config and self.data_config['load_bbox']:
                frame_dict.update(bbox_sequence[fid])

            if 'load_action' in self.data_config and self.data_config['load_action']:
                frame_dict.update(action_sequence[fid])

            frame_dict_sequence.append(frame_dict)
        
        frame_dict_sequence = self._preprocess_motion(frame_dict_sequence)

        return frame_dict_sequence

    
    def _preprocess_motion(self, frame_dict_sequence):
        for id, frame_dict in enumerate(frame_dict_sequence):
            if id == 0:
                curr_to_prev_lidar_rt = np.eye(4)
            else:
                cur_img_info = frame_dict['img_info']
                cur_global2lidar_rotation = cur_img_info['CAM_FRONT']['global2lidar_rotation']
                cur_global2lidar_translation = cur_img_info['CAM_FRONT']['global2lidar_translation']

                cur_global2lidar = np.eye(4)
                cur_global2lidar[:3, :3] = cur_global2lidar_rotation
                cur_global2lidar[:3, 3] = cur_global2lidar_translation

                prev_img_info = frame_dict_sequence[id-1]['img_info']
                prev_global2lidar_rotation = prev_img_info['CAM_FRONT']['global2lidar_rotation']
                prev_global2lidar_translation = prev_img_info['CAM_FRONT']['global2lidar_translation']
                prev_global2lidar = np.eye(4)
                prev_global2lidar[:3, :3] = prev_global2lidar_rotation
                prev_global2lidar[:3, 3] = prev_global2lidar_translation
                
                curr_to_prev_lidar_rt = prev_global2lidar @ np.linalg.inv(cur_global2lidar)
            
            frame_dict['curr_to_prev_lidar_rt'] = torch.from_numpy(curr_to_prev_lidar_rt).float()
        return frame_dict_sequence
    
    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
        else:
            worker_id = 0
            num_workers = 1
        
        num_workers = max(num_workers, 1)

        # lance_dataset = lance.torch.data.LanceDataset(
        #     self.data_config['dataset_path'],
        #     columns=["scenario_id", "scenario"],
        #     filter="raw_sensors_valid = true",
        #     batch_size=1,
        #     # batch_readahead=8,  # Control multi-threading reads.
        #     to_tensor_fn=to_tensor,
        #     sampler = ShardedFragmentSampler(
        #         rank=self.rank*num_workers+worker_id,  # Rank of the current dataloader thread
        #         world_size=self.world_size*num_workers,  # Total number of processes
        #         randomize=True,
        #     )
        # )
        lance_iterator = None

        if self.leaveout:
            world_size = self.world_size*num_workers+1
        else:
            world_size = self.world_size*num_workers
        
        while True:
            sample, lance_iterator = get_nextitem(lance_iterator, self.data_config['dataset_path'], rank=self.rank*num_workers+worker_id, world_size=world_size)


            [sample] = sample.to_pylist()
                
            sequence_data = self._preprocess_sample(sample)
            if sequence_data is None:
                # print(f"sequence_data is {sequence_data}", flush=True)
                continue
            
            try:
                if self.pipeline is not None:
                    sequence_data = [self.pipeline(item) for item in sequence_data]
            except (PIL.UnidentifiedImageError, OSError, FileNotFoundError):
                print(f"invalid data: {sample['scenario_id']}", flush=True)
                continue

            yield sequence_data
            del sequence_data

In [21]:
def _get_scenario_info(scenario):
    scenario_dict = {}
    timeofday = scenario['metadata']['timeofday']
    scenario_dict['timeofday'] = timeofday
    
    scenario_dict['location'] = scenario['metadata']['map']
    
    objects = scenario['metadata']['object_summary']
    sdc_id = scenario['metadata']['sdc_id']
    object_categories = []
    for obj_id in objects:
        if sdc_id != obj_id:
            obj_cat = objects[obj_id]['type'].lower()
            if obj_cat not in object_categories:
                object_categories.append(obj_cat)
    object_categories = ', '.join(object_categories)
    # scenario_dict['description'] = description

    if 'language_description' in scenario:
        scenario_dict['language_description'] = scenario['language_description']

    return scenario_dict

def _get_object_info(scenario, id):
    gt_bboxes_3d = []
    gt_names_3d = []
    
    sdc_id = scenario['metadata']['sdc_id']
    tracks = scenario['tracks']
    for obj_id in tracks:
        if obj_id == 'sdc_id':
            continue
        obj_info = tracks[obj_id]
        if not obj_info['state']['valid'][id]:
            continue
        position = obj_info['state']['position'][id]
        heading = obj_info['state']['heading'][id][None]
        velocity = obj_info['state']['velocity'][id]
        length = obj_info['state']['length'][id]
        width = obj_info['state']['width'][id]
        height = obj_info['state']['height'][id]
        
        obj_size = np.concatenate([length, width, height], axis=0)
        
        bbox = np.concatenate([position, obj_size, heading], axis=0)
        gt_bboxes_3d.append(bbox)

        label = obj_info['type']
        gt_names_3d.append(label)
    
    if len(gt_bboxes_3d) > 0:
        gt_bboxes_3d = np.stack(gt_bboxes_3d, axis=0)
    else:
        gt_bboxes_3d = np.zeros((0, 7))
    gt_names_3d = np.array(gt_names_3d)

    return {'gt_bboxes_3d': gt_bboxes_3d, 'gt_names_3d': gt_names_3d}


def _get_camera_info(camera_sensors):
    all_camera_info = {}
    
    for cam_name in camera_sensors:
        cam_data = camera_sensors[cam_name]
        cam_info = {}
        cam_info['data_path'] = cam_data['cam_abs_path'] # cam_abs_path in s3
        # cam_info['data_path'] = cam_info['data_path'].replace('s3://', '/media/')
        # cam_info['data_path'] = cam_info['data_path'].replace('s3://', '/media/')

        cam_info['cam_intrinsic'] = np.array(cam_data['cam_intrinsic'], dtype=np.float32)
        cam_info['sensor2lidar_translation'] = np.array(cam_data['sensor2lidar_translation'], dtype=np.float32)
        cam_info['sensor2lidar_rotation'] = np.array(cam_data['sensor2lidar_rotation'], dtype=np.float32)

        cam_info['ego2global_rotation'] = Quaternion(np.array(cam_data['ego2global_rotation'], dtype=np.float32)).rotation_matrix
        cam_info['ego2global_translation'] = np.array(cam_data['ego2global_translation'], dtype=np.float32)

        sensor2lidar = np.eye(4)
        sensor2lidar[:3, :3] = cam_info['sensor2lidar_rotation']
        sensor2lidar[:3, 3] = cam_info['sensor2lidar_translation']
        
        ego2global = np.eye(4)
        ego2global[:3, :3] = cam_info['ego2global_rotation']
        ego2global[:3, 3] = cam_info['ego2global_translation']
        
        sensor2ego = np.eye(4)
        sensor2ego[:3, :3] = Quaternion(np.array(cam_data['sensor2ego_rotation'], dtype=np.float32)).rotation_matrix
        sensor2ego[:3, 3] = np.array(cam_data['sensor2ego_translation'], dtype=np.float32)

        global2lidar = sensor2lidar @ np.linalg.inv(sensor2ego) @ np.linalg.inv(ego2global)
        cam_info['global2lidar_rotation'] = global2lidar[:3, :3]
        cam_info['global2lidar_translation'] = global2lidar[:3, 3]

        all_camera_info[cam_name] = cam_info
        

    return all_camera_info

In [22]:
scenario_dict = _get_scenario_info(scenario)
print(scenario_dict)

{'timeofday': '2018-07-24-11-22-45+0800', 'location': 'singapore-onenorth'}


In [23]:
raw_sensors = scenario['raw_sensors']
print("length", len(raw_sensors))

length 39


In [24]:
raw_sensor = raw_sensors[1]
print(raw_sensor.keys())

dict_keys(['images', 'lidar', 'metadata'])


In [25]:
print(raw_sensor['metadata'])

{'timestamp': 1532402928147847, 'description': "The vehicle is traveling on a clear, urban road during what appears to be daytime under sunny weather conditions. To the left-front, a large delivery truck is parked on the side of the road, partially obstructing the view of the sidewalk, which seems to be bordered by tall modern buildings with reflective glass facades. Directly in front, the road is marked by well-defined lanes, and it's relatively free of traffic, with only a few vehicles visible in the distance. The right-front view shows a pedestrian pathway and a neatly landscaped area with a fire hydrant visible, adjacent to more contemporary buildings. Behind the vehicle, the left-rear view reveals a sidewalk that continues alongside the buildings, while directly behind, the road is clear of traffic. The right-rear view also shows an unobstructed pathway with no vehicles or pedestrians in close proximity."}


In [26]:
print(raw_sensor['images'])

{'CAM_FRONT': {'cam_rel_path': './data/nuscenes/samples/CAM_FRONT/n015-2018-07-24-11-22-45+0800__CAM_FRONT__1532402928112460.jpg', 'cam_abs_path': 's3://research-datasets/nuscenes/samples/CAM_FRONT/n015-2018-07-24-11-22-45+0800__CAM_FRONT__1532402928112460.jpg', 'type': 'CAM_FRONT', 'sample_data_token': '4b6870ae200c4b969b91c50a9737f712', 'sensor2ego_translation': [np.float64(1.70079118954), np.float64(0.0159456324149), np.float64(1.51095763913)], 'sensor2ego_rotation': [np.float64(-0.499801543056913), np.float64(0.5030316162024876), np.float64(-0.49977981143868067), np.float64(0.4973708382454277)], 'ego2global_translation': [np.float64(409.8506686425138), np.float64(1176.9702106041582), np.float64(0.0)], 'ego2global_rotation': [np.float64(-0.5742377826385253), np.float64(0.0018667925555496466), np.float64(-0.014165885989800115), np.float64(0.8185638715152707)], 'timestamp': 1532402928112460, 'sensor2lidar_rotation': array([[ 0.00300787,  0.01917838,  0.99981155],
       [-0.99996501, 

In [27]:
print(raw_sensor['lidar'])

{'lidar_pc': '4f792c8da81e4cb7aca1790654da1c27', 'lidar2ego_translation': [np.float64(0.90735041686876), np.float64(0.008772958597376897), np.float64(0.3406964732839508)], 'lidar2ego_rotation': [np.float64(0.9999216977040046), np.float64(0.0029373022353292567), np.float64(0.012118719001139618), np.float64(0.0010523146349076737)], 'ego2global_translation': [np.float64(409.7431520488096), np.float64(1176.6769733781416), np.float64(0.0)], 'ego2global_rotation': [np.float64(-0.5744205119534879), np.float64(0.001328867575165046), np.float64(-0.01391702941565466), np.float64(0.8184409727343229)], 'log_center': array([ 411.24948141, 1180.74320966,    0.        ]), 'lidar_abs_path': 's3://research-datasets/nuscenes/samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402928147847.pcd.bin', 'lidar_rel_path': './data/nuscenes/samples/LIDAR_TOP/n015-2018-07-24-11-22-45+0800__LIDAR_TOP__1532402928147847.pcd.bin', 'description': "The vehicle is traveling on a clear, urban road during wha

In [28]:
print(raw_sensor['images'].keys())

dict_keys(['CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_FRONT_LEFT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT'])


In [29]:
camera = raw_sensor['images']['CAM_FRONT']
print(camera.keys())

dict_keys(['cam_rel_path', 'cam_abs_path', 'type', 'sample_data_token', 'sensor2ego_translation', 'sensor2ego_rotation', 'ego2global_translation', 'ego2global_rotation', 'timestamp', 'sensor2lidar_rotation', 'sensor2lidar_translation', 'cam_intrinsic'])


In [30]:
print(camera['cam_abs_path'])

s3://research-datasets/nuscenes/samples/CAM_FRONT/n015-2018-07-24-11-22-45+0800__CAM_FRONT__1532402928112460.jpg


In [31]:
mean=[0.5, 0.5, 0.5]
std=[0.5, 0.5, 0.5]

data_config={
    'dataset_path': dataset_path,
    'horizon': 4, # timesteps
    'interval': 2,
    'random_interval': 0,
    'first_frame': 0,
        
    'load_bbox': True,
    'object_range': (-50, -50, -10, 50, 50, 10),
    
    'load_action': False,
    # 'action_horizon': args.action_horizon,
    # 'kmeans_path': args.kmeans_path,
    
    'cams': ['CAM_FRONT', 'CAM_FRONT_RIGHT',  'CAM_BACK_RIGHT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_FRONT_LEFT'],
    'Ncams': 6,
    #TODO: support higher resolution and bigger model
    'input_size': (192, 336),
    'src_size': (1200, 2000),
    'keep_ratio': False,
    'mean': mean,
    'std': std,

    # Augmentation
    'resize': (0, 0),
    'rot': (0, 0),
    'flip': False,
    'crop_h': (0.0, 0.0),
    'resize_test':0.0,
}

class_names =[ 'car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'traffic_barrier', 'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone']
# class_names = [item.upper() for item in class_names]

load_keys = ['img_inputs', 'gt_bboxes_3d', 'gt_labels_3d']

train_pipeline = [
    pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=True),
    pipelines.DefaultFormatBundle3D(class_names=class_names, with_gt=True, with_label=True),
    pipelines.Collect3D(keys=load_keys),
]

nuscenes_dataset = NuScenesataWrapper(data_config, pipeline=train_pipeline)

In [32]:
sequence_data = nuscenes_dataset._get_frame_from_scenario(scenario)
print(sequence_data[2]['curr_to_prev_lidar_rt'])

tensor([[ 9.9986e-01, -1.6400e-02,  3.5045e-03,  8.2454e+00],
        [ 1.6413e-02,  9.9986e-01, -3.5697e-03,  6.3853e-02],
        [-3.4455e-03,  3.6267e-03,  9.9999e-01,  2.6949e-01],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00]])


In [33]:
print(sequence_data[0]['timeofday'])

2018-07-24


In [34]:
print(sequence_data[2]['description'])

pedestrian, barrier, traffic_cone, truck, bicycle, car, bus, motorcycle, construction_vehicle


In [35]:
sample = [nuscenes_dataset.pipeline(item) for item in sequence_data]
print(sample)

INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for s3 via

[{'img_metas': DataContainer({'sample_idx': 'scene-0061', 'curr_to_prev_lidar_rt': tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]]), 'description': 'pedestrian, barrier, traffic_cone, truck, bicycle, car, bus, motorcycle, construction_vehicle', 'location': 'singapore-onenorth', 'timeofday': '2018-07-24', 'timestep': np.float64(0.0)}), 'img_inputs': (tensor([[[[-0.7725, -0.8353, -0.8667,  ..., -0.0196, -0.0353, -0.0275],
          [-0.8039, -0.8431, -0.8902,  ..., -0.0118, -0.0118, -0.0275],
          [-0.8353, -0.8431, -0.8980,  ..., -0.0039, -0.0118, -0.0118],
          ...,
          [-0.5294, -0.4824, -0.4745,  ..., -0.1843, -0.2000, -0.1843],
          [-0.5529, -0.4902, -0.4902,  ..., -0.1608, -0.1686, -0.1843],
          [-0.5529, -0.5059, -0.5059,  ..., -0.1529, -0.1765, -0.2000]],

         [[-0.8275, -0.8510, -0.8196,  ...,  0.0902,  0.0745,  0.0824],
          [-0.8431, -0.8510, -0.8431,  ...,  0.0980,  0.0980,  0.0824],

In [36]:
def get_gt_images(sample):
    gt_images = []
    for tid in range(len(sample)):
        frame_sample = sample[tid]
        frame_images = frame_sample['img_inputs'][0].clone()

        frame_images = (frame_images + 1) / 2
        frame_images = (frame_images*255).to(torch.uint8)
        
        gt_images.append(frame_images)
    return gt_images

def visualize_box(sample, video_imgs, num_views=8):
    all_bbox_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas'].data
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas'].data
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }
        if num_views == 1:
            for key in camera_meta_datas:
                if key not in ['seq_ids', 'curr_to_prev_lidar']:
                    camera_meta_datas[key] = camera_meta_datas[key][:, 0:1]


        camera_meta_data_list.append(camera_meta_datas)       

    bbox_sequence = []
    for tid in range(len(sample)):
        frame_data = sample[tid]
        gt_bboxes_3d = frame_data['gt_bboxes_3d'].data
        gt_labels_3d = frame_data['gt_labels_3d'].data

        if len(gt_labels_3d) > 0:
            xyz = gt_bboxes_3d.bottom_center
            dims = gt_bboxes_3d.dims
            yaw = gt_bboxes_3d.yaw
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            xyz, yaw = convert_bbox(xyz, yaw, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
            boxes = torch.cat([xyz, dims, yaw[:, None]], dim=-1)
            gt_bboxes_3d = LiDARInstance3DBoxes(boxes)
            bbox_num = boxes.shape[0]
        else:
            gt_bboxes_3d = LiDARInstance3DBoxes(torch.zeros((0, 7)))
            gt_labels_3d = torch.zeros(0)
            bbox_num = 0
        
        gt_labels_3d = gt_labels_3d.to(torch.int32)
        bbox_sequence.append([gt_bboxes_3d, gt_labels_3d, bbox_num])

    for tid in range(len(sample)):
        frame_meta_data = camera_meta_data_list[tid]
        frame_bbox = bbox_sequence[tid]
        
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [PImage.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'gt_bboxes_3d': bbox_sequence[tid][0],
            'gt_labels_3d': bbox_sequence[tid][1],
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        bbox_imgs = draw_box_on_imgs(frame_data, frame_images, np.array(class_names))
        
        bbox_imgs = [torch.from_numpy(np.array(img)) for img in bbox_imgs]
        bbox_imgs = torch.stack(bbox_imgs, dim=0)
        bbox_imgs = bbox_imgs.permute(0, 3, 1, 2)
        all_bbox_imgs.append(bbox_imgs)
    
    return all_bbox_imgs


def save_imgs(images, suffix=''):
    TV3HW = torch.stack(images, dim=0)
    TV3HW = TV3HW.flatten(0, 1)
    vthw = torchvision.utils.make_grid(TV3HW, nrow=6, padding=0, pad_value=1.0)
    vthw = vthw.clone().permute(1, 2, 0).cpu().numpy()
    vthw = PImage.fromarray(vthw.astype(np.uint8))
    vthw.save(f"../vis/output_video_{suffix}.png")

In [37]:
gt_images = get_gt_images(sample)
gt_images_bbox = visualize_box(sample, gt_images)
save_imgs(gt_images_bbox, 'box')

In [38]:
print(len(gt_images_bbox))

4


In [39]:
for i in range(4):
    curr_to_prev_lidar_rt = sample[i]['img_metas'].data['curr_to_prev_lidar_rt']
    xy = curr_to_prev_lidar_rt[:3, 3]
    print(xy)

tensor([0., 0., 0.])
tensor([9.3003e+00, 6.0858e-03, 3.2748e-01])
tensor([8.2454, 0.0639, 0.2695])
tensor([7.2052e+00, 3.5805e-03, 2.6619e-01])


In [ ]:
print(sample[0]['gt_bboxes_3d'].data.bottom_center)

tensor([[-1.1232e+01, -3.0815e+01,  1.3205e+00],
        [ 4.6289e+01, -9.6098e+00,  2.1154e+00],
        [-1.0355e+01, -5.9050e+00,  2.0470e-01],
        [ 4.1973e+01, -2.1657e+01,  2.2945e+00],
        [ 1.5519e+01, -7.0906e+00,  1.1966e+00],
        [-9.2381e+00, -6.6218e+00,  4.8083e-01],
        [ 4.5768e+01, -6.7050e+00,  3.1782e+00],
        [ 2.1555e+01, -7.5189e+00,  1.3697e+00],
        [ 2.9553e+01, -8.0383e+00,  1.6182e+00],
        [-1.8308e+01, -3.6807e+01,  1.2178e+00],
        [ 2.7579e+01, -7.9601e+00,  1.5688e+00],
        [ 1.9536e+01, -7.3485e+00,  1.3302e+00],
        [ 7.2717e+00,  1.6073e+01,  2.1982e+00],
        [-1.3619e+01,  3.8430e+00,  1.3308e+00],
        [-1.5395e+01, -6.6346e+00,  8.1592e-02],
        [ 3.3741e+01, -9.1343e+00,  1.6584e+00],
        [ 1.9389e+01, -8.6393e+00,  1.2657e+00],
        [ 1.6856e+01,  2.5182e+00,  1.9034e+00],
        [ 1.5253e+01,  4.4986e+00,  3.6939e+00],
        [ 3.7622e+01, -2.0744e+01,  2.3034e+00],
        [ 3.5009e+01